In [89]:
def parse_transcript(transcipt_name):

    chat_stats = {}
    policy = None
    dialogs = {}
    DIALOG_TYPES = ['cts', 'hdc', 'faq']

    user_dialog_agent_mapping = {}
    user_dialog_nums = {}

    with open(transcipt_name, "r") as transcript:
        for line in transcript:
            if "(POLICY:" in line:
                current_dialog = {}
                tmp = line.split()
                user = tmp[1].strip()
                policy = tmp[3].strip(")").strip()
                user_dialog_agent_mapping[user] = policy
                current_dialog['user'] = user
                current_dialog['turns'] = []
                # TODO: add back goals to dialogs and analyze if there is anything interesting at a per/goal level
                # goal_type = tmp[-1].strip(")").strip()
            elif "USER:" in line and not "POST-NLU" in line:
                current_dialog["turns"].append(line)
            elif "SYSTEM" in line:
                current_dialog['turns'].append(line)
            elif "DIALOG END:" in line:
                current_dialog["end_condition"] = line.split(":")[1].strip()
            elif "SUBJECTIVE LENGTH" in line:
                current_dialog["sub_length"] = line.split(":")[1].strip()
            elif "SUBJECTIVE QUALITY" in line:
                current_dialog["sub_quality"] = line.split(":")[1].strip()
            elif line.strip() == "":
                # TODO: Change this line to analyse one group at a time
                if current_dialog and policy in DIALOG_TYPES:
                    obj_length = len(current_dialog["turns"])
                    if not policy in chat_stats:
                        chat_stats[policy] = {"length": [], "end_condition": [], "sub_length": [], "sub_quality": []}
                    chat_stats[policy]["length"].append(obj_length)
                    chat_stats[policy]["end_condition"].append(current_dialog["end_condition"])
                    chat_stats[policy]["sub_length"].append(int(current_dialog["sub_length"]))
                    chat_stats[policy]["sub_quality"].append(int(current_dialog["sub_quality"]))
                    current_dialog["length"] = obj_length
                    if current_dialog['user'] not in user_dialog_nums:
                        user_dialog_nums[current_dialog['user']] = 0
                    user_dialog_nums[current_dialog['user']] += 1
                    if policy not in dialogs:
                        dialogs[policy] = []
                    dialogs[policy].append(current_dialog)
                    current_dialog = {}
    return (chat_stats, dialogs, user_dialog_nums, user_dialog_agent_mapping)


In [90]:
chat_stats, dialogs, user_dialog_nums, user_agent_map = parse_transcript("combined/combined_transcript.txt")

### Distribution of User Style Preferences

In [91]:
user_style_map = {}

with open('combined/combined_survey_log.txt', "r") as survey_file:
    for line in survey_file:
        if "PREFERRED_STYLE" in line:
            user, style = line.split("||")
            user = user.split(":")[1].strip()
            style = style.split(":")[1].strip().lower()
            user_style_map[user] = style

In [92]:
# get distribution of user style preferences
preferred_styles = {}

users = set()
for policy in dialogs:
    preferred_styles[policy] = {}
    for dialog in dialogs[policy]:
        user = dialog["user"]
        if user not in users:
            style = user_style_map[user]
            if style not in preferred_styles[policy]:
                preferred_styles[policy][style] = 0
            preferred_styles[policy][style] += 1
            users.add(user)
for policy in preferred_styles:
    print(policy)
    print(dict(sorted(preferred_styles[policy].items())))

cts
{'base': 7, 'formal': 5, 'friendly': 4, 'personal': 6}
faq
{'base': 6, 'formal': 9, 'friendly': 5, 'personal': 2}
hdc
{'base': 7, 'formal': 10, 'friendly': 3, 'personal': 2}


#### Check that there is no significant difference in distribution of user preferred styles between dialog system conditions

In [93]:
preferred_styles = {}
styles = ["formal", "base", "personal", "friendly"]
users = set()
for policy in dialogs:
    preferred_styles[policy] = []
    for dialog in dialogs[policy]:
        user = dialog["user"]
        if user not in users:
            style = styles.index(user_style_map[user])
            preferred_styles[policy].append(style)
            users.add(user)

In [94]:
# Check that there are no significant differences
from scipy.stats import kruskal

cts = preferred_styles["cts"]
hdc = preferred_styles["hdc"]
faq = preferred_styles["faq"]

kruskal(cts, hdc, faq)

KruskalResult(statistic=2.6417239204752683, pvalue=0.26690514116646635)

### Get user demographic information

In [95]:
import csv
user_stats_per_policy = {}
user_exp_map = {}
with open('pre_survey_ling_ad.csv', 'r') as infile:
    reader = csv.DictReader(infile, delimiter="|")
    for row in reader:
        keep_keys = ['gender', 'age', 'experience_chatbots', 'experience_businesstravel', 'user']
        policy = user_agent_map[row['user']]
        if policy not in user_stats_per_policy:
            user_stats_per_policy[policy] = []
        exp = int(row['experience_chatbots'])
        user = row['user']
        user_exp_map[user] = exp
        user_stats_per_policy[policy].append({key: row[key] for key in keep_keys})

### Check if chatbot experience influences success 

In [96]:
from scipy.stats import chi2_contingency
import numpy as np

success_by_condition_and_exp = {}
for policy in dialogs:
    for dialog in dialogs[policy]:
        user = dialog['user']
        end_condtion = 1 if "SUCCESS" in dialog['end_condition'] else 0
        if user in user_exp_map:
            exp = user_exp_map[user]
        else:
            print(user)
            continue

        if not policy in success_by_condition_and_exp:
            success_by_condition_and_exp[policy] = {}
        if not exp in success_by_condition_and_exp[policy]:
            success_by_condition_and_exp[policy][exp] = []
        success_by_condition_and_exp[policy][exp].append(end_condtion)

successes = []
failures = []
for policy in success_by_condition_and_exp:
    print(policy)
    for exp in success_by_condition_and_exp[policy]:
        successes.append(sum(success_by_condition_and_exp[policy][exp]))
        failures.append(len(success_by_condition_and_exp[policy][exp]) - sum(success_by_condition_and_exp[policy][exp]))

    table = np.array([successes, failures])
    res = chi2_contingency(table)
    print(res.statistic)
    print(res.pvalue)
    

cts
3.852540442014126
0.27783366010193783
faq
23.888894794179308
0.002392168883998203
hdc
35.517770329028174
0.00038714712042740265


In [97]:
# get gender information
gender_distribution = {}
for policy in user_stats_per_policy:
    gender_distribution[policy] = {}
    for entry in user_stats_per_policy[policy]:
        gender = entry['gender']
        if gender not in gender_distribution[policy]:
            gender_distribution[policy][gender] = 0
        gender_distribution[policy][gender] += 1

print(gender_distribution)

{'cts': {'male': 11, 'female': 11}, 'faq': {'male': 11, 'female': 11}, 'hdc': {'female': 13, 'male': 8, 'other': 1}}


In [98]:
# get age information
age_distribution = {}
for policy in user_stats_per_policy:
    age_distribution[policy] = {}
    for entry in user_stats_per_policy[policy]:
        age = entry['age']
        if age not in age_distribution[policy]:
            age_distribution[policy][age] = 0
        age_distribution[policy][age] += 1

print(age_distribution)

{'cts': {'40-49': 5, '20-29': 11, '30-39': 5, '<20': 1}, 'faq': {'40-49': 2, '20-29': 15, '30-39': 4, '50-59': 1}, 'hdc': {'40-49': 2, '20-29': 11, '30-39': 9}}


In [99]:
# get previous chatbot experience information
chatbot_exp_distribution = {}
chatbot_avg_exp = []
for policy in user_stats_per_policy:
    chatbot_exp_distribution[policy] = {}
    for entry in user_stats_per_policy[policy]:
        chatbot_exp = int(entry['experience_chatbots'])
        chatbot_avg_exp.append(chatbot_exp)
        if chatbot_exp not in chatbot_exp_distribution[policy]:
            chatbot_exp_distribution[policy][chatbot_exp] = 0
        chatbot_exp_distribution[policy][chatbot_exp] += 1

print(chatbot_exp_distribution)
print(sum(chatbot_avg_exp)/len(chatbot_avg_exp))

{'cts': {3: 11, 2: 1, 4: 7, 5: 3}, 'faq': {3: 9, 4: 8, 5: 3, 2: 1, 1: 1}, 'hdc': {4: 9, 3: 10, 2: 2, 5: 1}}
3.484848484848485


In [100]:
# get previous business travel experience information
bt_exp_distribution = {}
bt_avg_exp = []
for policy in user_stats_per_policy:
    bt_exp_distribution[policy] = {}
    for entry in user_stats_per_policy[policy]:
        bt_exp = entry['experience_businesstravel']
        bt_avg_exp.append(int(bt_exp))
        if bt_exp not in bt_exp_distribution[policy]:
            bt_exp_distribution[policy][bt_exp] = 0
        bt_exp_distribution[policy][bt_exp] += 1

print(bt_exp_distribution)
print(sum(bt_avg_exp)/len(bt_avg_exp))

{'cts': {'3': 7, '1': 11, '2': 4}, 'faq': {'3': 10, '1': 5, '2': 4, '4': 3}, 'hdc': {'1': 7, '2': 8, '3': 7}}
2.106060606060606


**Result**: No significant difference between participants in terms of age, gender, business travel exp or chatbot exp

### Get demographic info for baseline

In [101]:
_, baseline_dialogs, _, baseline_user_agent_map = parse_transcript("../mental_models/combined_data/transcript.txt")

baseline_user_stats_per_policy = {}
baseline_user_exp_map = {}
with open('../mental_models/cts_combined/pre_survey_cts.csv', 'r') as infile:
    reader = csv.DictReader(infile, delimiter=";")
    for row in reader:
        keep_keys = ['gender', 'age', 'experience_chatbots', 'experience_businesstravel', 'user']
        user = row['user']
        policy = baseline_user_agent_map[user]
        if policy not in baseline_user_stats_per_policy:
            baseline_user_stats_per_policy[policy] = []
        baseline_user_stats_per_policy[policy].append({key: row[key] for key in keep_keys})
        exp = int(row['experience_chatbots'])
        baseline_user_exp_map[user] = exp        

with open('../mental_models/hdc_faq_combined/pre_survey.csv', 'r') as infile:
    reader = csv.DictReader(infile, delimiter="|")
    for row in reader:
        keep_keys = ['gender', 'age', 'experience_chatbots', 'experience_businesstravel', 'user']
        user = row['user']
        policy = baseline_user_agent_map[user]
        if policy not in baseline_user_stats_per_policy:
            baseline_user_stats_per_policy[policy] = []
        baseline_user_stats_per_policy[policy].append({key: row[key] for key in keep_keys})
        exp = int(row['experience_chatbots'])
        baseline_user_exp_map[user] = exp      

In [102]:
success_by_condition_and_exp = {}
for policy in baseline_dialogs:
    for dialog in baseline_dialogs[policy]:
        user = dialog['user']
        end_condtion = 1 if "SUCCESS" in dialog['end_condition'] else 0
        if user in baseline_user_exp_map:
            exp = baseline_user_exp_map[user]
        else:
            print(user)
            continue

        if not policy in success_by_condition_and_exp:
            success_by_condition_and_exp[policy] = {}
        if not exp in success_by_condition_and_exp[policy]:
            success_by_condition_and_exp[policy][exp] = []
        success_by_condition_and_exp[policy][exp].append(end_condtion)

successes = []
failures = []
for policy in success_by_condition_and_exp:
    print(policy)
    for exp in success_by_condition_and_exp[policy]:
        successes.append(sum(success_by_condition_and_exp[policy][exp]))
        failures.append(len(success_by_condition_and_exp[policy][exp]) - sum(success_by_condition_and_exp[policy][exp]))

    table = np.array([successes, failures])
    res = chi2_contingency(table)
    print(res.statistic)
    print(res.pvalue)

cts
4.567117924726514
0.20638108589913293
hdc
18.65197636336905
0.009350370202036329
faq
19.55026959686696
0.03380508142898756


In [103]:
def get_chatbot_exp(user_stats):
    chatbot_exp_distribution = {}
    for policy in user_stats:
        chatbot_exp_distribution[policy] = []
        for entry in user_stats[policy]:
            chatbot_exp = entry['experience_chatbots']
            chatbot_exp_distribution[policy].append(chatbot_exp)
    return chatbot_exp_distribution

In [81]:
from scipy.stats import f_oneway
baseline_chatbot_exp = get_chatbot_exp(baseline_user_stats_per_policy)
ling_ad_chatbot_exp = get_chatbot_exp(user_stats_per_policy)

baseline_exp = [int(i) for i in baseline_chatbot_exp['cts'] + baseline_chatbot_exp['hdc'] + baseline_chatbot_exp['hdc']]
ling_ad_exp = [int(i) for i in ling_ad_chatbot_exp['cts'] + ling_ad_chatbot_exp['hdc'] + ling_ad_chatbot_exp['faq']]

print(sum(baseline_exp), len(baseline_exp), sum(baseline_exp)/len(baseline_exp))
print(sum(ling_ad_exp), len(ling_ad_exp), sum(ling_ad_exp)/len(ling_ad_exp))

# f_oneway(baseline_chatbot_exp['cts'], baseline_chatbot_exp['hdc'], baseline_chatbot_exp['hdc'], ling_ad_chatbot_exp['cts'], ling_ad_chatbot_exp['hdc'], ling_ad_chatbot_exp['faq'])

192 65 2.953846153846154
230 66 3.484848484848485


**Notes**: There is a difference in chatbot experience, need to add to GAMM :(

In [82]:
user_chatbot_exp_mapping = {}
for policy in baseline_user_stats_per_policy:
    for entry in baseline_user_stats_per_policy[policy]:
        user = entry['user']
        exp = entry['experience_chatbots']
        user_chatbot_exp_mapping[user] = exp
    for entry in user_stats_per_policy[policy]:
        user = entry['user']
        exp = entry['experience_chatbots']
        user_chatbot_exp_mapping[user] = exp

In [104]:
import csv
with open("chat_stats.tsv", "w") as outfile:
    sanity_check = {"ling_ad": {}, "baseline": {}}
    writer = csv.writer(outfile, delimiter='\t',
                            quotechar='|', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(["dialog_system", "ling_style", "chatbot_exp", "length", "success", "sub_length", "sub_success"])
    for policy in dialogs:
        sanity_check['ling_ad'][policy] = []
        sanity_check['baseline'][policy] = []
        for dialog in dialogs[policy]:
            user = dialog['user']
            dialog_sytem = user_agent_map[user]
            ling_style = 'ling_ad'
            chatbot_exp = user_chatbot_exp_mapping[user]
            length = dialog['length']
            success = 1 if "SUCCESS" in dialog['end_condition'] else 0
            sanity_check['ling_ad'][policy].append(success)
            sub_length = dialog['sub_length']
            sub_quality = dialog['sub_quality']
            writer.writerow([dialog_sytem, ling_style, chatbot_exp, length, success, sub_length, sub_quality])

        for dialog in baseline_dialogs[policy]:
            user = dialog['user']
            dialog_sytem = baseline_user_agent_map[user]
            ling_style = 'baseline'
            chatbot_exp = user_chatbot_exp_mapping[user]
            length = dialog['length']
            success = 1 if "SUCCESS" in dialog['end_condition'] else 0
            sanity_check['baseline'][policy].append(success)
            sub_length = dialog['sub_length']
            sub_quality = dialog['sub_quality']
            writer.writerow([dialog_sytem, ling_style, chatbot_exp, length, success, sub_length, sub_quality])
    for condition in sanity_check:
        print(condition)
        for policy in sanity_check[condition]:
            print(policy, sum(sanity_check[condition][policy]), len(sanity_check[condition][policy]))


ling_ad
cts 57 64
faq 41 65
hdc 34 62
baseline
cts 47 61
faq 35 61
hdc 29 66


## Check for differences in success between different styles and dialog systems

In [85]:
from scipy.stats import barnard_exact

end_conditions = {}
for policy in dialogs:
    for dialog in dialogs[policy]:
        user = dialog['user']
        style = user_style_map[user]
        end_condition = dialog['end_condition']
        if not policy in end_conditions:
            end_conditions[policy] = {}
        if not style in end_conditions[policy]:
            end_conditions[policy][style] = []
        end_conditions[policy][style].append(1 if "SUCCESS" in end_condition else 0)


for policy in end_conditions:
    print(policy)
    success_per_style = {}
    for style in end_conditions[policy]:
        success_per_style[style] = sum(end_conditions[policy][style])/len(end_conditions[policy][style])
    print(success_per_style)
    print(barnard_exact(table=table, alternative="less"))

cts
{'base': 0.8571428571428571, 'personal': 0.8235294117647058, 'friendly': 0.9166666666666666, 'formal': 0.9285714285714286}
faq
{'personal': 0.6666666666666666, 'friendly': 0.3333333333333333, 'formal': 0.6923076923076923, 'base': 0.6666666666666666}
hdc
{'formal': 0.4827586206896552, 'friendly': 0.75, 'base': 0.55, 'personal': 0.2}


**Notes**: Note sure how much we can conclude here. 
- In the CTS condition, the styles all perform roughly equally
- In the faq, users who wanted a friendly interaction probably were not able to formulate their question in a way that it could be answered, reflects more on user than on system
- Not really enough users who chose friendly or personal to say too much about it, but it could be something to investigate. The easy language might have encouraged users to continue interacting when they normally would have ended. But the personal condition wound up so much worse... Alternative hypothesis, you can see somthing of user interaction preference from their choice of linguistic template (same people choose friendly who want longer dialogs, this is why they succeed in hdc and fail in faq)

### Check for dialog length

In [81]:
turns = {}
for policy in dialogs:
    for dialog in dialogs[policy]:
        user = dialog['user']
        style = user_style_map[user]
        turn_num = len(dialog['turns'])
        if not policy in turns:
            turns[policy] = {}
        if not style in turns[policy]:
            turns[policy][style] = []
        turns[policy][style].append(turn_num)

for policy in turns:
    print(policy)
    turns_per_style = {style: sum(turns[policy][style])/ len(turns[policy][style]) for style in turns[policy]}
    print(f_oneway(turns[policy]['formal'], turns[policy]['base'], turns[policy]['personal'], turns[policy]['friendly']))
    print(turns_per_style)

cts
F_onewayResult(statistic=1.7355368566792722, pvalue=0.16932359615258216)
{'base': 5.857142857142857, 'personal': 8.882352941176471, 'friendly': 7.25, 'formal': 3.642857142857143}
faq
F_onewayResult(statistic=40.152102654922224, pvalue=1.880954068090827e-14)
{'personal': 7.333333333333333, 'friendly': 2.1333333333333333, 'formal': 2.076923076923077, 'base': 2.4444444444444446}
hdc
F_onewayResult(statistic=0.8560416898285571, pvalue=0.4691198578858041)
{'formal': 11.827586206896552, 'friendly': 15.0, 'base': 13.8, 'personal': 7.6}
